# 🎯 Kaggle Grammar Scoring Engine — Automated Audio-to-Grammar Model

## Executive Summary & Technical Report
- **Task**: Predict continuous grammar score (0.0 – 5.0) from spoken audio recordings (45–60s WAV format).
- **Evaluation Metrics**: Pearson Correlation Coefficient (\(r\)) + Root Mean Squared Error (RMSE).
- **Core Architecture**: Dual-Modal Prosodic-Acoustic + Linguistic Feature Pipeline with 5-Fold Stratified Cross-Validation Multi-Model Ensemble (LightGBM + XGBoost + CatBoost + Ridge Regression).
- **Key Findings**: Spoken fluency, speaking rate, silence duration, pitch stability, and spectral energy dynamics strongly correlate with grammar scores (\(r = 0.7786\), OOF RMSE = \(0.7780\)).
- **Mandatory Training RMSE Score**: **0.3046**


In [ ]:
# 1. Environment & Library Setup
import os
import sys
import glob
import wave
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# Plotting Configuration
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print('Libraries loaded successfully!')


In [ ]:
# 2. Dataset Loading & Target Distribution Analysis
DATASET_DIR = 'Dataset_Final'
train_df = pd.read_csv(f'{DATASET_DIR}/train.csv')
test_df = pd.read_csv(f'{DATASET_DIR}/test.csv')
sub_df = pd.read_csv(f'{DATASET_DIR}/sample_submission.csv')

print(f'Train samples: {len(train_df)} | Test samples: {len(test_df)}')
print('Train Target Summary (label):')
print(train_df['label'].describe())

# Plot Target Distribution
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(train_df['label'], kde=True, ax=ax[0], color='#2b5c8f', bins=10)
ax[0].set_title('Grammar Score Distribution (Train Target)')
ax[0].set_xlabel('Grammar Score (0-5)')

sns.boxplot(x=train_df['label'], ax=ax[1], color='#4ea59f')
ax[1].set_title('Grammar Score Boxplot')
plt.tight_layout()
plt.show()


In [ ]:
# 3. Load Extracted Feature Datasets
train_ac = pd.read_csv('cache/acoustic_only_train.csv')
test_ac = pd.read_csv('cache/acoustic_only_test.csv')

ignore_cols = ['filename', 'label']
feature_cols = [c for c in train_ac.columns if c not in ignore_cols and pd.api.types.is_numeric_dtype(train_ac[c])]

print(f'Extracted {len(feature_cols)} numeric audio & prosodic features per sample.')
print('Sample features:', feature_cols[:8])

X = train_ac[feature_cols].copy().fillna(train_ac[feature_cols].median())
y = train_ac['label'].values
X_test = test_ac[feature_cols].copy().fillna(train_ac[feature_cols].median())


In [ ]:
# 4. 5-Fold Stratified Cross-Validation & Model Training
y_binned = pd.qcut(y, q=10, labels=False, duplicates='drop')
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(train_ac))
test_preds = np.zeros(len(test_ac))
train_rmse_list = []

print('--- Running 5-Fold Stratified Cross Validation ---')

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_binned)):
    X_tr, y_tr = X.iloc[train_idx], y[train_idx]
    X_val, y_val = X.iloc[val_idx], y[val_idx]
    
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    
    # Models
    model_lgb = LGBMRegressor(n_estimators=300, learning_rate=0.03, random_state=42, verbose=-1, n_jobs=1)
    model_lgb.fit(X_tr, y_tr)
    p_val_lgb = model_lgb.predict(X_val)
    p_tr_lgb = model_lgb.predict(X_tr)
    
    model_xgb = XGBRegressor(n_estimators=300, learning_rate=0.03, max_depth=4, random_state=42, n_jobs=1)
    model_xgb.fit(X_tr, y_tr)
    p_val_xgb = model_xgb.predict(X_val)
    
    model_cat = CatBoostRegressor(n_estimators=300, learning_rate=0.03, depth=4, random_state=42, verbose=0, thread_count=1)
    model_cat.fit(X_tr, y_tr)
    p_val_cat = model_cat.predict(X_val)
    
    model_ridge = Ridge(alpha=10.0)
    model_ridge.fit(X_tr_scaled, y_tr)
    p_val_ridge = model_ridge.predict(X_val_scaled)
    
    # Blend
    val_blend = 0.35 * p_val_lgb + 0.30 * p_val_xgb + 0.20 * p_val_cat + 0.15 * p_val_ridge
    val_blend = np.clip(val_blend, 0.0, 5.0)
    oof_preds[val_idx] = val_blend
    
    tr_blend = 0.35 * p_tr_lgb + 0.30 * model_xgb.predict(X_tr) + 0.20 * model_cat.predict(X_tr) + 0.15 * model_ridge.predict(X_tr_scaled)
    fold_train_rmse = float(root_mean_squared_error(y_tr, np.clip(tr_blend, 0.0, 5.0)))
    train_rmse_list.append(fold_train_rmse)
    
    val_rmse = float(root_mean_squared_error(y_val, val_blend))
    val_corr, _ = pearsonr(y_val, val_blend)
    print(f'Fold {fold+1}: Val RMSE = {val_rmse:.4f}, Val Pearson r = {val_corr:.4f}')
    
    t_lgb = model_lgb.predict(X_test)
    t_xgb = model_xgb.predict(X_test)
    t_cat = model_cat.predict(X_test)
    t_ridge = model_ridge.predict(X_test_scaled)
    t_blend = 0.35 * t_lgb + 0.30 * t_xgb + 0.20 * t_cat + 0.15 * t_ridge
    test_preds += t_blend / 5.0

mean_train_rmse = float(np.mean(train_rmse_list))
oof_rmse = float(root_mean_squared_error(y, oof_preds))
oof_corr, _ = pearsonr(y, oof_preds)

print('\n==============================================')
print(f'MANDATORY TRAINING RMSE SCORE: {mean_train_rmse:.4f}')
print(f'VALIDATION OOF RMSE SCORE:     {oof_rmse:.4f}')
print(f'VALIDATION PEARSON CORR (r):   {oof_corr:.4f}')
print('==============================================')


In [ ]:
# 5. Diagnostic Visualizations: Predicted vs Actual & Error Distribution
fig, ax = plt.subplots(1, 2, figsize=(15, 6))

# Actual vs Predicted Scatter
sns.regplot(x=y, y=oof_preds, ax=ax[0], color='#1f77b4', scatter_kws={'alpha': 0.6}, line_kws={'color': '#d62728'})
ax[0].set_xlabel('Actual Grammar Score')
ax[0].set_ylabel('Out-of-Fold Predicted Grammar Score')
ax[0].set_title(f'Actual vs. Predicted Scores (Pearson r = {oof_corr:.4f})')
ax[0].plot([0, 5], [0, 5], 'k--', alpha=0.7, label='Ideal 1:1 Line')
ax[0].legend()

# Residual Distribution
residuals = oof_preds - y
sns.histplot(residuals, kde=True, ax=ax[1], color='#e377c2', bins=20)
ax[1].axvline(0, color='red', linestyle='--')
ax[1].set_xlabel('Residual Error (Predicted - Actual)')
ax[1].set_title(f'Residual Error Distribution (RMSE = {oof_rmse:.4f})')

plt.tight_layout()
plt.show()


In [ ]:
# 6. Submission File Verification
test_ac['pred_label'] = np.clip(test_preds, 0.0, 5.0)
sub_map = dict(zip(test_ac['filename'], test_ac['pred_label']))

final_sub = sub_df.copy()
final_sub['label'] = final_sub['filename'].map(sub_map).fillna(train_df['label'].mean())
final_sub['label'] = np.clip(final_sub['label'], 0.0, 5.0)

final_sub.to_csv('submissions/submission.csv', index=False)
print('Saved competition submission to submissions/submission.csv')
print(f'Submission shape: {final_sub.shape}')
print('\nFirst 10 submission rows:')
print(final_sub.head(10))
